In [1]:
print("Hwllow")

Hwllow


In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ============================================================
# CONFIG
# ============================================================
DATA_DIR        = "/Users/tiarasabrina/Documents/PROJECT/AI/xlm-bert_bigru/code/"
BEST_MODEL_PATH = "/Users/tiarasabrina/Documents/PROJECT/AI/xlm-bert_bigru/best_model.pt"
MODEL_NAME      = "xlm-roberta-base"
MAX_LEN         = 64
MAX_MSGS        = 15
BATCH_SIZE      = 4
EPOCHS          = 5
LR              = 2e-5
POS_WEIGHT      = 10.0 
THRESHOLD       = 0.3  

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device     : {DEVICE}")
print(f"POS_WEIGHT : {POS_WEIGHT}")
print(f"THRESHOLD  : {THRESHOLD}")
df_en = pd.read_parquet(DATA_DIR + "df_train_en.parquet")
df_id = pd.read_parquet(DATA_DIR + "df_train_id.parquet")

for df in [df_en, df_id]:
    df['conv_id']     = df['conv_id'].astype(str)
    df['author']      = df['author'].astype(str)
    df['text']        = df['text'].fillna('').astype(str)
    df['is_predator'] = df['is_predator'].astype(int)

print(f"\nEN : {df_en['conv_id'].nunique()} convs | {len(df_en)} messages")
print(f"ID : {df_id['conv_id'].nunique()} convs | {len(df_id)} messages")

def split_convs(df):
    conv_labels = df.groupby('conv_id')['is_predator'].max().reset_index()
    train_c, temp_c = train_test_split(
        conv_labels, test_size=0.30,
        random_state=42, stratify=conv_labels['is_predator']
    )
    val_c, test_c = train_test_split(
        temp_c, test_size=0.50,
        random_state=42, stratify=temp_c['is_predator']
    )
    return (
        df[df['conv_id'].isin(train_c['conv_id'])].reset_index(drop=True),
        df[df['conv_id'].isin(val_c['conv_id'])].reset_index(drop=True),
        df[df['conv_id'].isin(test_c['conv_id'])].reset_index(drop=True)
    )

en_train, en_val, en_test = split_convs(df_en)
id_train, id_val, id_test = split_convs(df_id)

df_train = pd.concat([en_train, id_train], ignore_index=True)
df_val   = pd.concat([en_val,   id_val],   ignore_index=True)
df_test  = pd.concat([en_test,  id_test],  ignore_index=True)

print(f"\nTrain : {df_train['conv_id'].nunique()} convs | {len(df_train)} messages")
print(f"Val   : {df_val['conv_id'].nunique()} convs | {len(df_val)} messages")
print(f"Test  : {df_test['conv_id'].nunique()} convs | {len(df_test)} messages")

print(f"\nTrain label dist:\n{df_train.groupby('conv_id')['is_predator'].max().value_counts()}")
print(f"Val   label dist:\n{df_val.groupby('conv_id')['is_predator'].max().value_counts()}")
print(f"Test  label dist:\n{df_test.groupby('conv_id')['is_predator'].max().value_counts()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ConversationDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, max_msgs):
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.max_msgs  = max_msgs
        self.convs     = []

        for conv_id, group in df.groupby('conv_id'):
            msgs  = group['text'].tolist()[:max_msgs]
            label = int(group['is_predator'].max())
            self.convs.append((msgs, label))

    def __len__(self):
        return len(self.convs)

    def __getitem__(self, idx):
        msgs, label     = self.convs[idx]
        input_ids       = []
        attention_masks = []

        for msg in msgs:
            enc = self.tokenizer(
                msg,
                max_length=self.max_len,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            input_ids.append(enc['input_ids'].squeeze(0))
            attention_masks.append(enc['attention_mask'].squeeze(0))

        while len(input_ids) < self.max_msgs:
            input_ids.append(torch.zeros(self.max_len, dtype=torch.long))
            attention_masks.append(torch.zeros(self.max_len, dtype=torch.long))

        return {
            'input_ids'      : torch.stack(input_ids),
            'attention_mask' : torch.stack(attention_masks),
            'num_msgs'       : torch.tensor(len(msgs)),
            'label'          : torch.tensor(label, dtype=torch.float)
        }

print("\nBuilding datasets...")
train_dataset = ConversationDataset(df_train, tokenizer, MAX_LEN, MAX_MSGS)
val_dataset   = ConversationDataset(df_val,   tokenizer, MAX_LEN, MAX_MSGS)
test_dataset  = ConversationDataset(df_test,  tokenizer, MAX_LEN, MAX_MSGS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

class XLMRoBERTaBiGRU(nn.Module):
    def __init__(self, model_name, hidden_size=128, freeze_layers=10):
        super().__init__()
        self.xlm = AutoModel.from_pretrained(model_name)

        for i, layer in enumerate(self.xlm.encoder.layer):
            if i < freeze_layers:
                for param in layer.parameters():
                    param.requires_grad = False

        self.bigru = nn.GRU(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.attention  = nn.Linear(hidden_size * 2, 1)
        self.classifier = nn.Linear(hidden_size * 2, 1)

    def forward(self, input_ids, attention_mask, num_msgs):
        batch_size, max_msgs, max_len = input_ids.shape

        input_ids_flat      = input_ids.view(-1, max_len)
        attention_mask_flat = attention_mask.view(-1, max_len)

        outputs        = self.xlm(input_ids=input_ids_flat, attention_mask=attention_mask_flat)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        cls_embeddings = cls_embeddings.view(batch_size, max_msgs, 768)

        gru_out, _ = self.bigru(cls_embeddings)

        attn_scores  = self.attention(gru_out).squeeze(-1)
        mask         = torch.arange(max_msgs, device=input_ids.device).unsqueeze(0) < num_msgs.unsqueeze(1)
        attn_scores  = attn_scores.masked_fill(~mask, float('-inf'))
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)

        context = (gru_out * attn_weights).sum(dim=1)
        logit   = self.classifier(context).squeeze(-1)
        return logit

print("\nLoading XLM-RoBERTa...")
model = XLMRoBERTaBiGRU(MODEL_NAME).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

pos_weight = torch.tensor([POS_WEIGHT]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR)

def evaluate(loader, desc="Evaluating"):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc, leave=False):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            num_msgs       = batch['num_msgs'].to(DEVICE)
            labels         = batch['label']

            logits = model(input_ids, attention_mask, num_msgs)
            probs  = torch.sigmoid(logits).cpu().numpy()
            preds  = (probs >= THRESHOLD).astype(int)

            all_labels.extend(labels.numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)

    has_both_classes = len(set(all_labels)) > 1
    return {
        'accuracy'  : accuracy_score(all_labels, all_preds),
        'precision' : precision_score(all_labels, all_preds, zero_division=0),
        'recall'    : recall_score(all_labels, all_preds, zero_division=0),
        'f1'        : f1_score(all_labels, all_preds, zero_division=0),
        'auc_roc'   : roc_auc_score(all_labels, all_probs) if has_both_classes else 0.0
    }

best_val_f1 = 0
history     = []

print("\n" + "="*60)
print("TRAINING START")
print("="*60)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch in pbar:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        num_msgs       = batch['num_msgs'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask, num_msgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_loss    = total_loss / len(train_loader)
    val_metrics = evaluate(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")

    history.append({'epoch': epoch+1, 'loss': avg_loss, **val_metrics})

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Loss      : {avg_loss:.4f}")
    print(f"  Val Acc   : {val_metrics['accuracy']:.4f}")
    print(f"  Val Prec  : {val_metrics['precision']:.4f}")
    print(f"  Val Rec   : {val_metrics['recall']:.4f}")
    print(f"  Val F1    : {val_metrics['f1']:.4f}")
    print(f"  Val AUC   : {val_metrics['auc_roc']:.4f}")

    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  ✓ Best model saved! (F1={best_val_f1:.4f})")

print("\n" + "="*60)
print("TEST EVALUATION")
print("="*60)

# Fallback kalau ga ada best model (semua epoch F1=0)
if os.path.exists(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
    print("Loaded best model")
else:
    print("No best model saved, using last epoch model")

test_metrics = evaluate(test_loader, desc="Testing")

print(f"Accuracy  : {test_metrics['accuracy']:.4f}")
print(f"Precision : {test_metrics['precision']:.4f}")
print(f"Recall    : {test_metrics['recall']:.4f}")
print(f"F1-Score  : {test_metrics['f1']:.4f}")
print(f"AUC-ROC   : {test_metrics['auc_roc']:.4f}")

pd.DataFrame(history).to_csv(DATA_DIR + "training_history.csv", index=False)
print(f"\nTraining history saved to {DATA_DIR}training_history.csv")

Device     : mps
POS_WEIGHT : 10.0
THRESHOLD  : 0.3

EN : 500 convs | 7465 messages
ID : 500 convs | 7503 messages

Train : 700 convs | 10882 messages
Val   : 150 convs | 2186 messages
Test  : 150 convs | 1900 messages

Train label dist:
is_predator
0    673
1     27
Name: count, dtype: int64
Val   label dist:
is_predator
0    144
1      6
Name: count, dtype: int64
Test  label dist:
is_predator
0    144
1      6
Name: count, dtype: int64

Building datasets...
Train batches : 175
Val batches   : 38
Test batches  : 38

Loading XLM-RoBERTa...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4029.25it/s]
[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable parameters: 207,855,106

TRAINING START


Epoch 1/5 [Train]: 100%|██████████| 175/175 [04:14<00:00,  1.45s/it, loss=0.3100]



Epoch 1/5
  Loss      : 0.8174
  Val Acc   : 0.7267
  Val Prec  : 0.1277
  Val Rec   : 1.0000
  Val F1    : 0.2264
  Val AUC   : 0.9757
  ✓ Best model saved! (F1=0.2264)


...

Epoch 4/5 [Train]: 100%|██████████| 175/175 [04:27<00:00,  1.53s/it, loss=0.2054]



Epoch 4/5
  Loss      : 0.5189
  Val Acc   : 0.6933
  Val Prec  : 0.1154
  Val Rec   : 1.0000
  Val F1    : 0.2069
  Val AUC   : 0.9086


Epoch 5/5 [Train]: 100%|██████████| 175/175 [04:13<00:00,  1.45s/it, loss=0.1376]



Epoch 5/5
  Loss      : 0.4542
  Val Acc   : 0.9067
  Val Prec  : 0.3000
  Val Rec   : 1.0000
  Val F1    : 0.4615
  Val AUC   : 0.9688
  ✓ Best model saved! (F1=0.4615)

TEST EVALUATION
Loaded best model


Accuracy  : 0.9133
Precision : 0.2941
Recall    : 0.8333
F1-Score  : 0.4348
AUC-ROC   : 0.9306

Training history saved to /Users/tiarasabrina/Documents/PROJECT/AI/xlm-bert_bigru/code/training_history.csv
